# 01 — Binance Historical Data Download
## Téléchargement complet de toutes les données BTCUSDT depuis data.binance.vision

**Durée estimée** : 24-36h pour les aggTrades spot (~90 GB Parquet), 2-3h pour le reste.

**Flux** : `ZIP (Binance) → RAM → CSV parse → Parquet → GCS → ZIP supprimé`

**Sections** :
1. aggTrades spot (2017-08 → aujourd'hui) — PRIORITÉ 1
2. Klines 1s/1m/5m/1h spot — PRIORITÉ 1-2
3. bookTicker spot — PRIORITÉ 2
4. Futures (aggTrades, klines 1m, funding rate) — PRIORITÉ 2
5. Validation croisée aggTrades vs klines 1m

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Imports et configuration
# ════════════════════════════════════════════════════════════════════════
import os, sys, time, json
sys.path.insert(0, "/workspace")

import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, clear_output
import ipywidgets as widgets
from datetime import datetime

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.config import Config
from btc_pipeline.collectors.binance_historical import (
    run_binance_download, download_and_upload_month, generate_months,
    run_all_binance_tasks,
)

os.environ.setdefault("GCS_BUCKET_NAME",       "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND",        "gcs")
os.environ.setdefault("WORKSPACE_DIR",         "/workspace")

storage = StorageClient()
config = Config()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Vérifications préliminaires
# ════════════════════════════════════════════════════════════════════════
import shutil
_, _, free = shutil.disk_usage(os.environ.get("WORKSPACE_DIR", "/workspace"))
print(f"✅ Espace disque libre : {free/1e9:.1f} GB")

state = storage.get_pipeline_state()
print(f"✅ Pipeline state chargé")
for task_name, task_data in state.get("tasks", {}).items():
    if "binance" in task_name:
        print(f"   {task_name}: {task_data}")

## Section 1 — aggTrades Spot (PRIORITÉ 1)
C'est la source de données principale. ~90 GB Parquet sur GCS.
Chaque mois est téléchargé en un seul ZIP, parsé en mémoire, converti en Parquet et uploadé.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Section 1 : aggTrades spot (2017-08 → aujourd'hui)
# ════════════════════════════════════════════════════════════════════════
months = generate_months("2017-08-17")
print(f"📊 aggTrades spot : {len(months)} mois à vérifier")

# Widget de progression
progress = widgets.IntProgress(
    value=0, max=len(months),
    description="aggTrades:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)
status_label = widgets.HTML(value="Démarrage...")
display(widgets.VBox([progress, status_label]))

t_start = time.time()

def update_progress(current, total, info):
    progress.value = current
    elapsed = time.time() - t_start
    rate = current / max(1, elapsed) * 3600  # months/hour
    eta = (total - current) / max(0.01, rate)  # hours remaining
    status_label.value = (
        f"<b>{info['year']}-{info['month']:02d}</b> — "
        f"Status: {info['status']} — "
        f"{info.get('rows', 0):,} rows — "
        f"ETA: {eta:.1f}h"
    )

stats_aggtrades = run_binance_download(
    storage,
    market="spot",
    data_type="aggTrades",
    symbol="BTCUSDT",
    start_date="2017-08-17",
    progress_callback=update_progress,
)

duration = time.time() - t_start
print(f"\n✅ aggTrades spot terminé en {duration/3600:.1f}h")
print(f"   Complétés: {stats_aggtrades['completed']}, Skipped: {stats_aggtrades['skipped']}")
print(f"   Volume: {stats_aggtrades['total_gb']:.2f} GB")
print(f"   Échoués: {stats_aggtrades['failed_months']}")

## Section 2 — Klines Spot (1m, 5m, 1h)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Section 2 : Klines spot — 1m (validation), 5m, 1h
# ════════════════════════════════════════════════════════════════════════
kline_intervals = [
    ("1m", "2017-08-17"),
    ("5m", "2017-08-17"),
    ("1h", "2017-08-17"),
    ("1s", "2020-01-01"),  # 1s disponible depuis 2020
]

for interval, start_date in kline_intervals:
    print(f"\n▶ Klines {interval} spot...")
    stats = run_binance_download(
        storage,
        market="spot",
        data_type="klines",
        symbol="BTCUSDT",
        interval=interval,
        start_date=start_date,
    )
    print(f"  ✅ {interval}: {stats['completed']} mois, {stats['total_gb']:.2f} GB")

## Section 3 — bookTicker Spot

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Section 3 : bookTicker spot (best bid/ask)
# ════════════════════════════════════════════════════════════════════════
print("▶ bookTicker spot...")
stats_bt = run_binance_download(
    storage,
    market="spot",
    data_type="bookTicker",
    symbol="BTCUSDT",
    start_date="2020-01-01",
)
print(f"✅ bookTicker: {stats_bt['completed']} mois, {stats_bt['total_gb']:.2f} GB")

## Section 4 — Futures (aggTrades, klines 1m, funding rate)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Section 4 : Données Futures
# ════════════════════════════════════════════════════════════════════════
futures_tasks = [
    ("aggTrades", None, "2019-09-01"),
    ("klines", "1m", "2019-09-01"),
    ("fundingRate", None, "2019-09-01"),
]

for data_type, interval, start_date in futures_tasks:
    label = f"{data_type}" + (f" {interval}" if interval else "")
    print(f"\n▶ Futures {label}...")
    stats = run_binance_download(
        storage,
        market="futures/um",
        data_type=data_type,
        symbol="BTCUSDT",
        interval=interval,
        start_date=start_date,
    )
    print(f"  ✅ {label}: {stats['completed']} mois, {stats['total_gb']:.2f} GB")

## Section 5 — Validation croisée aggTrades vs Klines 1m

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Section 5 : Validation croisée
# ════════════════════════════════════════════════════════════════════════
from btc_pipeline.processors.temporal_aligner import (
    load_aggtrades_month, load_klines_month, validate_klines_vs_aggtrades,
)
from btc_pipeline.processors.bucket_aggregator import process_aggtrades_month

# Valider un mois récent
test_year, test_month = 2024, 1

print(f"Validation pour {test_year}-{test_month:02d}...")
df_agg = load_aggtrades_month(storage, test_year, test_month)
if not df_agg.empty:
    df_1s = process_aggtrades_month(df_agg)
    df_klines = load_klines_month(storage, test_year, test_month, "1m")
    if not df_klines.empty:
        result = validate_klines_vs_aggtrades(df_1s, df_klines)
        print(f"Résultat: {json.dumps(result, indent=2)}")
    else:
        print("⚠️  Klines 1m non disponibles pour ce mois")
else:
    print("⚠️  aggTrades non disponibles — télécharger d'abord")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Résumé final
# ════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 55)
print("RÉSUMÉ — BINANCE HISTORICAL")
print("═" * 55)

state = storage.get_pipeline_state()
total_gb = 0
for task_name, task_data in state.get("tasks", {}).items():
    if "binance" in task_name:
        print(f"  {task_name}: last={task_data.get('last_completed_month', '?')}")

# Compter les fichiers sur GCS
for prefix in ["raw/binance/spot_aggtrades", "raw/binance/spot_klines_1m",
               "raw/binance/futures_aggtrades", "raw/binance/futures_funding"]:
    files = storage.list_files(prefix)
    print(f"  {prefix}: {len(files)} files")

print("═" * 55)
storage.save_pipeline_state(state)
print("✅ État sauvegardé")